# EMode

[EMode](https://emodephotonix.com) is a commercial photonic simulation package with an electromagnetic waveguide mode solver based on the Finite Difference Method (FDM), eigenmode expansion propagation (EME), nonlinear photonics, and thermal/electrical FEM solvers.

The `emode` plugin translates gdsfactory cross-sections and layer stacks into EMode geometry, and gives you access to the full EMode API from the same session. See the [EMode documentation](https://docs.emodephotonix.com) for all available functions.

```{note}
Running this notebook requires a local EMode installation and license (a [free trial](https://emodephotonix.com/free-trial/) is available) in addition to `pip install gplugins[emode]`. The pip-installable `emodeconnection` client only provides the connection to the EMode application.
```

In [ ]:
from gdsfactory.cross_section import rib
from gdsfactory.gpdk import LAYER_STACK
from gdsfactory.technology import LayerStack

from gplugins.emode import EMode

## Define the geometry

We start from the generic PDK layer stack and keep only the layers relevant for a silicon-on-insulator (SOI) rib waveguide: a 220 nm core with a 90 nm slab, on a buried oxide, with an oxide cladding. All dimensions are in um, following the gdsfactory convention.

In [ ]:
layer_stack = LayerStack(
    layers={
        k: LAYER_STACK.layers[k].model_copy()
        for k in ("core", "clad", "slab90", "box")
    }
)

layer_stack.layers["core"].thickness = 0.22
layer_stack.layers["core"].zmin = 0.0

layer_stack.layers["slab90"].thickness = 0.09
layer_stack.layers["slab90"].zmin = 0.0

layer_stack.layers["box"].thickness = 1.5
layer_stack.layers["box"].zmin = -1.5

layer_stack.layers["clad"].thickness = 1.5
layer_stack.layers["clad"].zmin = 0.0

## Connect to EMode

Creating an `EMode` instance launches the EMode application and checks out a license.

In [ ]:
em = EMode(simulation_name="soi_rib")

## Build the waveguide

`build_waveguide` translates the cross-section and layer stack into EMode shapes and forwards the remaining settings to EMode. Dimensional settings are given in um and converted to nm (EMode's default units) automatically. Material names from the layer stack are matched against the EMode material database.

In [ ]:
em.build_waveguide(
    cross_section=rib(width=0.6),
    layer_stack=layer_stack,
    wavelength=1.55,
    num_modes=1,
    x_resolution=0.010,
    y_resolution=0.010,
    window_width=3.0,
    window_height=3.0,
    background_material="Air",
    max_effective_index=2.631,
)

## Solve for the modes

Any EMode function can be called as a method on the session. Launch the FDM mode solver:

In [ ]:
em.FDM()

Display the effective indices, TE fractions, and core confinement:

In [ ]:
em.report()

Plot the field and refractive index profiles:

In [ ]:
em.plot()

## Close the session

This saves the simulation file and releases the license.

In [ ]:
em.close()